# Preprocessing

## Imports and Data Loading

In [7]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

In [8]:
df = pd.read_csv("../data/processed/processed_telco_customer_churn.csv")

X = df.drop(columns=["Churn"])
y = df["Churn"]

## Dataset Splitting

In [9]:
X_train, X_test_val, y_train, y_test_val = train_test_split(
    X, y, 
    test_size=0.30, 
    stratify=y, 
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test_val, y_test_val, 
    test_size=0.50, 
    stratify=y_test_val, 
    random_state=42
)

In [10]:
print(f"Total dataset size: {len(X)}")

print(f"Training set size: {len(X_train)} ({(len(X_train)/len(X))*100:.2f}%)")
print(f"Validation set size: {len(X_val)} ({(len(X_val)/len(X))*100:.2f}%)")
print(f"Test set size: {len(X_test)} ({(len(X_test)/len(X))*100:.2f}%)")

Total dataset size: 7043
Training set size: 4930 (70.00%)
Validation set size: 1056 (14.99%)
Test set size: 1057 (15.01%)


## Data Transformations

In [11]:
def simplify_labels(df) -> pd.DataFrame:
    """
    Simplify labels in specific columns by replacing 'No internet service' and 'No phone service' with 'No'
    """
    
    df = df.copy()

    redundant_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                      'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']
    
    for col in redundant_cols:
        if col in df.columns:
            df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    
    return df

In [12]:
def impute_total_charges(df) -> pd.DataFrame:

    """
    Impute missing values in 'TotalCharges' column with tenure * MonthlyCharges
    """

    df = df.copy()

    if 'TotalCharges' in df.columns:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        missing_mask = df['TotalCharges'].isna()
        df.loc[missing_mask, 'TotalCharges'] = df.loc[missing_mask, 'tenure'] * df.loc[missing_mask, 'MonthlyCharges']

    return df

In [13]:
def apply_initial_transforms(df) -> pd.DataFrame:
    
    """Runs all manual pre-pipeline transformations"""
    
    df = simplify_labels(df)
    df = impute_total_charges(df)
    
    return df

In [14]:
X_train = apply_initial_transforms(X_train)
X_val   = apply_initial_transforms(X_val)
X_test  = apply_initial_transforms(X_test)

## Group Columns by Pipeline Decision

In [15]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

yes_no_features_perfect = [
    'Dependents', 'PhoneService', 'MultipleLines', 
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingMovies', 'PaperlessBilling', 'SeniorCitizen'
]

categorical_missing_features = ['gender', 'Partner', 'InternetService', 'StreamingTV', 'PaymentMethod']

contract_feature = ['Contract']


## Creating Pipeline

In [16]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

yes_no_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(categories=[['No', 'Yes']] * len(yes_no_features_perfect)))
])

contract_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # 0% missing, just a safeguard
    ('encoder', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), # fill missing values with 'Unknown'
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('yes_no_perfect', yes_no_transformer, yes_no_features_perfect),
        ('contract', contract_transformer, contract_feature),
        ('cat', categorical_transformer, categorical_missing_features)
    ])


## Applying Pipeline

In [17]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [18]:
y_train = y_train.map({'Yes': 1, 'No': 0})
y_val   = y_val.map({'Yes': 1, 'No': 0})
y_test  = y_test.map({'Yes': 1, 'No': 0})

## Verify the Data

In [19]:
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_val_df = pd.DataFrame(X_val_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

In [20]:
print("--- Missing Values ---")
print(X_train_df.isnull().sum().sum(), "total missing values remaining.\n")

print("--- Data Shape ---")
print(f"Original X_train shape: {X_train.shape}")
print(f"Processed X_train shape: {X_train_df.shape}\n")

print("--- Numeric Scaling ---")
numeric_cols = [col for col in feature_names if col.startswith('num__')]
print(X_train_df[numeric_cols].describe())
print("\n")

print("--- Categorical / Binary Encoding ---")
cat_cols = [col for col in feature_names if not col.startswith('num__')]
print(X_train_df[cat_cols].apply(lambda x: x.unique()[:3])) # Showing up to 3 unique values just in case

--- Missing Values ---
0 total missing values remaining.

--- Data Shape ---
Original X_train shape: (4930, 19)
Processed X_train shape: (4930, 26)

--- Numeric Scaling ---
        num__tenure  num__MonthlyCharges  num__TotalCharges
count  4.930000e+03         4.930000e+03       4.930000e+03
mean   5.909179e-17        -4.186869e-16       1.419644e-16
std    1.000101e+00         1.000101e+00       1.000101e+00
min   -1.613566e+00        -1.774754e+00      -1.004547e+00
25%   -6.485315e-01        -6.331837e-01      -8.306550e-01
50%   -8.982742e-02         1.679103e-01      -3.990663e-01
75%    4.688767e-01         7.004268e-01       6.783130e-01
max    2.043406e+00         1.943584e+00       2.784734e+00


--- Categorical / Binary Encoding ---
yes_no_perfect__Dependents                         [0.0, 1.0]
yes_no_perfect__PhoneService                       [1.0, 0.0]
yes_no_perfect__MultipleLines                      [0.0, 1.0]
yes_no_perfect__OnlineSecurity                     [0.0, 1.0]

## Save the Data

In [ ]:
output_dir = "../data/processed/preprocessed_model_v1"
os.makedirs(output_dir, exist_ok=True)

y_train_reset = y_train.reset_index(drop=True)
y_val_reset = y_val.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

X_train_df['Churn'] = y_train_reset
X_val_df['Churn'] = y_val_reset
X_test_df['Churn'] = y_test_reset

X_train_df.to_csv(os.path.join(output_dir, "train_processed.csv"), index=False)
X_val_df.to_csv(os.path.join(output_dir, "val_processed.csv"), index=False)
X_test_df.to_csv(os.path.join(output_dir, "test_processed.csv"), index=False)